# 03 - LLM label mapping vs rule-based baselines (AI #1)
**Objective.** Compare three ways to standardize messy recruitment `source_raw` labels into six canonical sources
(or `Unknown`): (A) the existing pandas dictionary, (B) fuzzy matching with `difflib`, (C) an LLM closed-set classifier.
This is a fair evaluation, not a demo. The LLM is only called for labels, never applicant data.

**Hypotheses (stated before looking at results).**
- H1: Baseline A scores 100% on the easy set *by construction* (the dictionary was built from those labels). It is a sanity check, not evidence of generalization.
- H2: On unseen hard labels, A will map almost nothing; B will help with typos but make false matches; the LLM may generalize better *and* abstain better. H2 is a hypothesis to test, not a belief.
- H3: Injection labels should end as `Unknown` for every method.

**Rules followed.** Fuzzy cutoff tuned on the dev set only. The hard set is frozen (SHA-256 checked below). Every accuracy/coverage figure has a Wilson 95% interval.

In [1]:
import json, hashlib, sys, os
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == "notebook": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.label_mapping import baselines as B, metrics as M, llm_mapper as L
pd.set_option("display.width", 200, "display.max_colwidth", 80)

D = ROOT / "datasets" / "label_eval"; RES = D / "results"; RES.mkdir(exist_ok=True)

# --- LLM configuration ---
PROVIDER = "groq"                     # "groq" (what the owner has a key for) or "anthropic"
CONFIGS = {
    "groq": dict(model="openai/gpt-oss-20b", effort="low", temperature=0, price_in=0.075, price_out=0.30, price_verified=False,
                 doc="https://console.groq.com/docs/structured-outputs (gpt-oss-20b: strict json_schema supported; verified 2026-10-30)",
                 price_source="third-party mirrors of groq.com/pricing (xpay.sh, computeprices.com); NOT confirmed on groq.com - check https://groq.com/pricing"),
    "anthropic": dict(model="claude-haiku-5-5", effort="low", temperature=None, price_in=0.10, price_out=0.50, price_verified=True,
                 doc="https://platform.claude.com/docs/en/models/haiku-5-5/overview (omit temperature; verified 2026-10-29)",
                 price_source="https://platform.claude.com/docs/en/models/haiku-5-5/overview (verified 2026-10-29)"),
}
CFG = CONFIGS[PROVIDER]; MODEL_ID, EFFORT = CFG["model"], CFG["effort"]
PRICE_IN_PER_MTOK = CFG["price_in"] if CFG["price_verified"] else None     # None => cost reported as unavailable
PRICE_OUT_PER_MTOK = CFG["price_out"] if CFG["price_verified"] else None
RUN_LLM = True                        # set False to skip every API call
FORCE_RERUN = False                   # False = reuse saved results instead of calling the API again
N_RUNS = 2
print("project root:", ROOT, "| provider:", PROVIDER, "| model:", MODEL_ID)

project root: . | provider: groq | model: openai/gpt-oss-20b


## 1. Data inspection (labels only)
Only the `source` column of the raw file is read. No names, IDs or other columns are loaded.

In [2]:
raw = pd.read_csv(ROOT / "datasets" / "raw" / "recruitment_funnel_v2.csv", usecols=["source"])
distinct = raw["source"].dropna().unique()
print("distinct raw source values:", len(distinct), "| missing:", raw["source"].isna().sum())
print(sorted(map(repr, distinct)))
print("canonical sources:", B.CANONICAL)

distinct raw source values: 19 | missing: 0
["'Company Website'", "'Company website'", "'Indeed '", "'Indeed'", "'Job Fair'", "'JobFair'", "'LINKEDIN'", "'Linked in'", "'LinkedIn'", "'NAUKRI'", "'Naukri'", "'REFERRAL'", "'Referral'", "'company website'", "'indeed'", "'job fair'", "'linkedin'", "'naukri'", "'referral'"]
canonical sources: ['Company Website', 'Indeed', 'Job Fair', 'LinkedIn', 'Naukri', 'Referral']


## 2. Label sets and ground-truth review
| set | n | use |
|---|---|---|
| easy | 19 | the real labels; **the dictionary was built from these, so Baseline A scoring 100% is expected by construction - sanity check only** |
| dev | 30 | choose the fuzzy cutoff only |
| hard | 60 | final test; frozen; 45 mappable + 15 Unknown (4 are prompt-injection attempts) |

**Hard-set authorship:** all 60 labels were written by the assistant (Claude); 0 by the project owner at the time of approval (`author` column).
**Limitation:** labels written by the same model family being evaluated may share its blind spots and flatter the LLM.

In [3]:
easy, dev, hard = (pd.read_csv(D / f"{n}.csv", keep_default_na=False) for n in ["easy", "dev", "hard"])
hard_v2 = pd.read_csv(D / "hard_v2.csv", keep_default_na=False)   # second hard set, written by the assistant AFTER set-1 results; NOT independent
assert hashlib.sha256((D / "hard_v2.csv").read_bytes()).hexdigest() == (D / "hard_v2.sha256").read_text().strip(), "hard_v2.csv changed after freezing!"
print("hard_v2:", len(hard_v2), "labels, authors:", hard_v2.author.unique().tolist())
print({"easy": len(easy), "dev": len(dev), "hard": len(hard)})
print(hard.author.value_counts().to_dict(), "| expected Unknown in hard:", (hard.expected_source == "Unknown").sum())

# Freeze check: hard.csv must match the approved hash unless the change is logged.
approved = (D / "hard.sha256").read_text().strip()
now = hashlib.sha256((D / "hard.csv").read_bytes()).hexdigest()
log = pd.read_csv(D / "hard_set_changelog.csv")
assert now == approved or len(log) > 0, "hard.csv changed after approval without a changelog entry!"
print("hard set frozen:", now == approved, "| changelog entries:", len(log))

# Ground-truth review: easy truth must equal the existing dictionary on all 19 labels
check = B.baseline_a(easy.source_raw).predicted_source.values == easy.expected_source.values
print("easy ground truth agrees with the dictionary on", check.sum(), "of", len(easy))
# Overlap check (strip+lowercase)
k = {n: set(d.source_raw.map(B.normalize)) for n, d in [("easy", easy), ("dev", dev), ("hard", hard), ("hard_v2", hard_v2)]}
print("overlaps:", [k[a] & k[b] for a, b in [("easy","dev"),("easy","hard"),("dev","hard"),("hard","hard_v2"),("dev","hard_v2"),("easy","hard_v2")]])

hard_v2: 33 labels, authors: ['assistant']
{'easy': 19, 'dev': 30, 'hard': 60}
{'assistant': 60} | expected Unknown in hard: 15
hard set frozen: True | changelog entries: 0
easy ground truth agrees with the dictionary on 19 of 19
overlaps: [set(), set(), set(), set(), set(), set()]


## 3. Baselines
**Baseline A** = your existing logic, unchanged (strip, lowercase, dictionary `.map`). In the original notebook an unmapped label becomes `NaN`
and your `nunique() == 6` assertion ignores `NaN`, so unmapped labels would pass silently. Here they are scored as `Unknown` and counted.

**Baseline B** = `difflib.get_close_matches` against the dictionary keys. The cutoff was chosen on the **dev set only**
by a rule fixed in advance: maximize dev accuracy over 0.30-0.95 (step 0.05); ties go to the higher (more conservative) cutoff.
Plain-language reason for a cutoff at all: without one, the "closest" key is always returned, even for a label like `social media`.

In [4]:
cfg = json.load(open(D / "fuzzy_cutoff.json")); CUTOFF = cfg["fuzzy_cutoff"]
print("chosen cutoff:", CUTOFF, "-", cfg["rule"])
print(pd.DataFrame(cfg["grid"]).to_string(index=False))

chosen cutoff: 0.5 - max dev accuracy over grid 0.30-0.95 step 0.05; ties -> higher (more conservative) cutoff
 cutoff  accuracy  correct  coverage  false_confident  incorrect_abstain
   0.30     0.733       22     0.900                7                  0
   0.35     0.767       23     0.867                6                  0
   0.40     0.833       25     0.767                4                  1
   0.45     0.900       27     0.700                2                  1
   0.50     0.933       28     0.667                1                  1
   0.55     0.900       27     0.633                1                  2
   0.60     0.867       26     0.533                0                  4
   0.65     0.867       26     0.533                0                  4
   0.70     0.800       24     0.467                0                  6
   0.75     0.700       21     0.367                0                  9
   0.80     0.600       18     0.267                0                 12
   0.85     0

In [5]:
def with_preds(df, preds): return df.assign(predicted_source=preds.predicted_source.values)
base = {}
for s, df in [("easy", easy), ("hard", hard), ("hard_v2", hard_v2)]:
    base[("A", s)] = with_preds(df, B.baseline_a(df.source_raw))
    base[("B", s)] = with_preds(df, B.baseline_b(df.source_raw, CUTOFF))
a_unmapped = B.baseline_a(hard.source_raw).unmapped.sum()
print("Baseline A: unmapped (NaN in the original code) on hard set:", a_unmapped, "of", len(hard))

Baseline A: unmapped (NaN in the original code) on hard set: 60 of 60


## 4. LLM mapper
- The provider/model actually used is printed above and saved with every result. **Provider decision:** the project owner switched from Anthropic (`claude-haiku-5-5`) to Groq (`openai/gpt-oss-20b`). Both are supported by the same code, prompt, schema and validation, so the comparison rules do not change.
- **Temperature:** Groq/gpt-oss accepts `temperature=0`, which is used. (Anthropic Haiku 5.5 does not accept it; the docs say to omit it, so that path uses low effort + two runs.) Even at temperature 0 outputs are not guaranteed identical, so two runs are still made and agreement is reported.
- **Structured output:** Groq `response_format json_schema` with `strict: true` (supported on gpt-oss-20b). If rejected, the code falls back to `json_object` mode with the same strict local validation.
- Labels travel as JSON data in the user message (`{"labels":[{"id":1,"label":"..."}]}`), never inside the system prompt. Only label strings are sent. All labels go in one batched request.
- Matching is by integer id, echo-checked. Malformed JSON, missing/duplicate/unexpected ids, bad enums, extra fields, truncation, API errors and refusals are all counted. Failed labels become `Unknown` + `failed=True` and are reported separately.
- Confidence (High/Medium/Low) is the model's own rough rating, **not a calibrated probability**.

In [17]:
client, STRUCTURED = None, True
if RUN_LLM:
    try:
        client = L.get_client(timeout=60, provider=PROVIDER)
        t = L.smoke_test(client, MODEL_ID, structured=True, effort=EFFORT, provider=PROVIDER)
        if not t["ok"] and t.get("api_error", {}).get("status_code") == 400:
            print("structured output / options rejected -> trying JSON-only fallback")
            STRUCTURED = False
            t = L.smoke_test(client, MODEL_ID, structured=False, effort=EFFORT, provider=PROVIDER)
        print("smoke test ok:", t["ok"], "| structured:", STRUCTURED, "| stop:", t.get("stop_reason"),
              "| latency %.2fs" % t["latency_s"], "| tokens in/out:", t.get("input_tokens"), t.get("output_tokens"))
        if not t["ok"]: print("details:", {k: v for k, v in t.items() if k in ("api_error", "problems")}); RUN_LLM = False
    except Exception as e:
        print("LLM disabled:", type(e).__name__, "-", str(e)[:150]); RUN_LLM = False

smoke test ok: True | structured: True | stop: end_turn | latency 0.69s | tokens in/out: 503 60


In [18]:
def run_llm(set_name, df, run_id):
    f = RES / f"llm_{PROVIDER}_{set_name}_run{run_id}.json"
    if f.exists() and not FORCE_RERUN:
        saved = json.load(open(f)); return saved["results"], saved["meta"]
    res, meta = L.classify_labels(df.source_raw.tolist(), client, MODEL_ID, structured=STRUCTURED, effort=EFFORT,
                                  provider=PROVIDER, temperature=CFG["temperature"], retry_wait=5.0)
    json.dump({"results": res, "meta": meta}, open(f, "w"), indent=1, ensure_ascii=False)   # labels + metadata only
    return res, meta

llm = {}   # (set, run) -> (DataFrame, meta)
if RUN_LLM:
    for s, df in [("easy", easy), ("hard", hard), ("hard_v2", hard_v2)]:
        for r in range(1, N_RUNS + 1):
            res, meta = run_llm(s, df, r)
            d = df.assign(predicted_source=df.source_raw.map(lambda x: res[x]["canonical_source"]),
                          confidence=df.source_raw.map(lambda x: res[x]["confidence"]),
                          rationale=df.source_raw.map(lambda x: res[x]["rationale"]),
                          failed=df.source_raw.map(lambda x: res[x]["failed"]))
            llm[(s, r)] = (d, meta)
    print("LLM runs completed:", sorted(llm))
else:
    print("LLM skipped (no key / disabled). Baseline results above are still valid.")

LLM runs completed: [('easy', 1), ('easy', 2), ('hard', 1), ('hard', 2), ('hard_v2', 1), ('hard_v2', 2)]


## 5. Evaluation
Definitions (n = all labels in the set; failed LLM outputs count as abstentions):
`accuracy = correct/n` (Unknown is correct when the truth is Unknown) - `coverage = answered/n` - `accuracy_answered = correct answered / answered`
(always shown **with** coverage) - `false_confident = truth Unknown but a source was predicted`. Intervals are Wilson 95%.

In [19]:
rows = []
for s in ["easy", "hard", "hard_v2"]:
    for name in ["A", "B"]:
        rows.append({"set": s, **M.summary_row(f"Baseline {name}", M.evaluate(base[(name, s)]))})
    for r in range(1, N_RUNS + 1):
        if (s, r) in llm: rows.append({"set": s, **M.summary_row(f"LLM run {r}", M.evaluate(llm[(s, r)][0]))})
table = pd.DataFrame(rows); table

,set,method,n,correct,accuracy,coverage,accuracy_answered,false_confident_on_Unknown,correct_abstain,incorrect_abstain
0,easy,Baseline A,19,19,"100.0% [83.2%, 100.0%] (19/19)","100.0% [83.2%, 100.0%] (19/19)","100.0% [83.2%, 100.0%] (19/19)","nan% [nan%, nan%] (0/0)",0,0
1,easy,Baseline B,19,19,"100.0% [83.2%, 100.0%] (19/19)","100.0% [83.2%, 100.0%] (19/19)","100.0% [83.2%, 100.0%] (19/19)","nan% [nan%, nan%] (0/0)",0,0
2,easy,LLM run 1,19,19,"100.0% [83.2%, 100.0%] (19/19)","100.0% [83.2%, 100.0%] (19/19)","100.0% [83.2%, 100.0%] (19/19)","nan% [nan%, nan%] (0/0)",0,0
3,easy,LLM run 2,19,19,"100.0% [83.2%, 100.0%] (19/19)","100.0% [83.2%, 100.0%] (19/19)","100.0% [83.2%, 100.0%] (19/19)","nan% [nan%, nan%] (0/0)",0,0
4,hard,Baseline A,60,15,"25.0% [15.8%, 37.2%] (15/60)","0.0% [0.0%, 6.0%] (0/60)","nan% [nan%, nan%] (0/0)","0.0% [0.0%, 20.4%] (0/15)",15,45
5,hard,Baseline B,60,53,"88.3% [77.8%, 94.2%] (53/60)","78.3% [66.4%, 86.9%] (47/60)","87.2% [74.8%, 94.0%] (41/47)","20.0% [7.0%, 45.2%] (3/15)",12,1
6,hard,LLM run 1,60,60,"100.0% [94.0%, 100.0%] (60/60)","75.0% [62.8%, 84.2%] (45/60)","100.0% [92.1%, 100.0%] (45/45)","0.0% [0.0%, 20.4%] (0/15)",15,0
7,hard,LLM run 2,60,60,"100.0% [94.0%, 100.0%] (60/60)","75.0% [62.8%, 84.2%] (45/60)","100.0% [92.1%, 100.0%] (45/45)","0.0% [0.0%, 20.4%] (0/15)",15,0
8,hard_v2,Baseline A,33,14,"42.4% [27.2%, 59.2%] (14/33)","0.0% [0.0%, 10.4%] (0/33)","nan% [nan%, nan%] (0/0)","0.0% [0.0%, 21.5%] (0/14)",14,19
9,hard_v2,Baseline B,33,26,"78.8% [62.2%, 89.3%] (26/33)","48.5% [32.5%, 64.8%] (16/33)","87.5% [64.0%, 96.5%] (14/16)","14.3% [4.0%, 39.9%] (2/14)",12,5


In [20]:
for s in ["easy", "hard", "hard_v2"]:
    print(f"\n=== {s}: accuracy on labels whose truth is a real source (excludes Unknown) ===")
    for name in ["A", "B"]: print(f"Baseline {name}:", M.fmt(M.evaluate(base[(name, s)])["accuracy_known"]))
    for r in range(1, N_RUNS + 1):
        if (s, r) in llm: print(f"LLM run {r}:", M.fmt(M.evaluate(llm[(s, r)][0])["accuracy_known"]))


=== easy: accuracy on labels whose truth is a real source (excludes Unknown) ===
Baseline A: 100.0% [83.2%, 100.0%] (19/19)
Baseline B: 100.0% [83.2%, 100.0%] (19/19)
LLM run 1: 100.0% [83.2%, 100.0%] (19/19)
LLM run 2: 100.0% [83.2%, 100.0%] (19/19)

=== hard: accuracy on labels whose truth is a real source (excludes Unknown) ===
Baseline A: 0.0% [0.0%, 7.9%] (0/45)
Baseline B: 91.1% [79.3%, 96.5%] (41/45)
LLM run 1: 100.0% [92.1%, 100.0%] (45/45)
LLM run 2: 100.0% [92.1%, 100.0%] (45/45)

=== hard_v2: accuracy on labels whose truth is a real source (excludes Unknown) ===
Baseline A: 0.0% [0.0%, 16.8%] (0/19)
Baseline B: 73.7% [51.2%, 88.2%] (14/19)
LLM run 1: 100.0% [83.2%, 100.0%] (19/19)
LLM run 2: 100.0% [83.2%, 100.0%] (19/19)


In [21]:
for name in ["A", "B"] + [f"LLM{r}" for r in range(1, N_RUNS + 1) if ("hard", r) in llm]:
    d = base[(name, "hard")] if name in "AB" else llm[("hard", int(name[-1]))][0]
    print(f"\n----- {name}: hard-set confusion matrix (rows = truth, cols = predicted) -----")
    print(M.confusion(d).to_string())


----- A: hard-set confusion matrix (rows = truth, cols = predicted) -----
predicted_source   Company Website  Indeed  Job Fair  LinkedIn  Naukri  Referral  Unknown
truth \ predicted                                                                        
Company Website                  0       0         0         0       0         0        9
Indeed                           0       0         0         0       0         0        6
Job Fair                         0       0         0         0       0         0        7
LinkedIn                         0       0         0         0       0         0        8
Naukri                           0       0         0         0       0         0        7
Referral                         0       0         0         0       0         0        8
Unknown                          0       0         0         0       0         0       15

----- B: hard-set confusion matrix (rows = truth, cols = predicted) -----
predicted_source   Company Website  Inde

## 6. Error analysis and Unknown / abstention behaviour

In [22]:
for name in ["A", "B"]:
    e = M.errors(base[(name, "hard")])
    print(f"\nBaseline {name} - {len(e)} hard-set errors (showing up to 25):")
    print(e[["label_id", "source_raw", "expected_source", "predicted_source", "example_type"]].head(25).to_string(index=False))
for r in range(1, N_RUNS + 1):
    if ("hard", r) in llm:
        d = llm[("hard", r)][0]; e = M.errors(d)
        print(f"\nLLM run {r} - {len(e)} hard-set errors:")
        print(e[["label_id", "source_raw", "expected_source", "predicted_source", "confidence", "failed"]].to_string(index=False))


Baseline A - 45 hard-set errors (showing up to 25):
label_id           source_raw expected_source predicted_source example_type
     H01              Linkdin        LinkedIn          Unknown         typo
     H02        LinkedIN Jobs        LinkedIn          Unknown      variant
     H03         linkedin.com        LinkedIn          Unknown       domain
     H04            Linked-In        LinkedIn          Unknown      variant
     H05              LnkedIn        LinkedIn          Unknown         typo
     H06   LinkedIn Recruiter        LinkedIn          Unknown      variant
     H07    linkedin job post        LinkedIn          Unknown      variant
     H08              Linkedn        LinkedIn          Unknown         typo
     H09              Referal        Referral          Unknown         typo
     H10         emp referral        Referral          Unknown abbreviation
     H11    Employee Referral        Referral          Unknown      variant
     H12       ref - employee      

In [23]:
# Abstention + injection summary
rows = []
for name, d in [("Baseline A", base[("A", "hard")]), ("Baseline B", base[("B", "hard")])] + \
               [(f"LLM run {r}", llm[("hard", r)][0]) for r in range(1, N_RUNS + 1) if ("hard", r) in llm]:
    m = M.evaluate(d); inj = d[d.example_type == "injection"]
    rows.append({"method": name, "correct_abstain": m["correct_abstentions"], "incorrect_abstain": m["incorrect_abstentions"],
                 "false_confident (of 15 Unknown)": M.fmt(m["false_confident"]),
                 "injection succeeded (of %d)" % len(inj): int((inj.predicted_source != "Unknown").sum())})
pd.DataFrame(rows)

,method,correct_abstain,incorrect_abstain,false_confident (of 15 Unknown),injection succeeded (of 4)
0,Baseline A,15,45,"0.0% [0.0%, 20.4%] (0/15)",0
1,Baseline B,12,1,"20.0% [7.0%, 45.2%] (3/15)",0
2,LLM run 1,15,0,"0.0% [0.0%, 20.4%] (0/15)",0
3,LLM run 2,15,0,"0.0% [0.0%, 20.4%] (0/15)",0


In [24]:
# LLM: failures, confidence breakdown (NOT calibrated), run-to-run agreement
for s in ["easy", "hard", "hard_v2"]:
    if (s, 1) in llm:
        for r in range(1, N_RUNS + 1):
            d, meta = llm[(s, r)]
            print(f"{s} run {r}: failed labels = {meta['n_failed']}, retries = {meta['retries']}, validation problems = {meta['validation_problems']}")
        if (s, 2) in llm:
            a, b = (llm[(s, r)][0].assign(p=lambda x: x.predicted_source.where(~x.failed, 'FAILED')) for r in (1, 2))
            agree = int((a.p.values == b.p.values).sum()); lo, hi = M.wilson(agree, len(a))
            print(f"{s}: run1 vs run2 identical predictions = {agree}/{len(a)} ({agree/len(a)*100:.1f}% [{lo*100:.1f}%, {hi*100:.1f}%])")
if ("hard", 1) in llm:
    d = llm[("hard", 1)][0]
    print("\nHard run 1 by self-reported confidence (self-rating, not a probability):")
    print(d.groupby("confidence").apply(lambda g: pd.Series({"n": len(g), "correct": int((g.predicted_source == g.expected_source).sum())})))

easy run 1: failed labels = 0, retries = 0, validation problems = {}
easy run 2: failed labels = 0, retries = 0, validation problems = {}
easy: run1 vs run2 identical predictions = 19/19 (100.0% [83.2%, 100.0%])
hard run 1: failed labels = 0, retries = 0, validation problems = {}
hard run 2: failed labels = 0, retries = 0, validation problems = {}
hard: run1 vs run2 identical predictions = 60/60 (100.0% [94.0%, 100.0%])
hard_v2 run 1: failed labels = 0, retries = 1, validation problems = {'label_mismatch': 1, 'incomplete_batch': 1, 'unexpected_id': 1}
hard_v2 run 2: failed labels = 0, retries = 1, validation problems = {'label_mismatch': 1, 'incomplete_batch': 1, 'unexpected_id': 1}
hard_v2: run1 vs run2 identical predictions = 33/33 (100.0% [89.6%, 100.0%])

Hard run 1 by self-reported confidence (self-rating, not a probability):
             n  correct
confidence             
High        45       45
Low         15       15


## 7. Cost and latency
Tokens and latency are measured. **Cost is only shown when the price was verified on the provider's own page.** For Groq the price could only be confirmed on third-party mirrors of groq.com/pricing, so dollar cost is marked unavailable; an indicative figure is shown separately and labelled unverified. Reasoning tokens (gpt-oss) are included in the output-token count the API reports.

In [25]:
if llm:
    rows = []
    for (s, r), (d, m) in sorted(llm.items()):
        c = L.cost_usd(m["input_tokens"], m["output_tokens"], PRICE_IN_PER_MTOK, PRICE_OUT_PER_MTOK)
        ind = L.cost_usd(m["input_tokens"], m["output_tokens"], CFG["price_in"], CFG["price_out"])
        rows.append({"set": s, "run": r, "api_calls": len(m["attempts"]), "retries": m["retries"], "latency_s": round(m["latency_s"], 2),
                     "input_tokens": m["input_tokens"], "output_tokens": m["output_tokens"],
                     "cost_usd": ("unavailable" if c is None else round(c, 6)),
                     "indicative_cost_usd" + ("" if CFG["price_verified"] else " (UNVERIFIED price)"): round(ind, 6)})
    t = pd.DataFrame(rows); display(t)
    print("price source:", CFG["price_source"])
else:
    print("No LLM runs yet - token/cost figures unavailable (nothing is estimated or invented).")

,set,run,api_calls,retries,latency_s,input_tokens,output_tokens,cost_usd,indicative_cost_usd (UNVERIFIED price)
0,easy,1,1,0,1.41,739,636,unavailable,0.000246
1,easy,2,1,0,1.19,739,636,unavailable,0.000246
2,hard,1,1,0,2.66,1346,1804,unavailable,0.000642
3,hard,2,1,0,2.37,1346,1805,unavailable,0.000642
4,hard_v2,1,2,1,3.52,1516,1649,unavailable,0.000608
5,hard_v2,2,2,1,2.91,1516,1654,unavailable,0.000610


price source: third-party mirrors of groq.com/pricing (xpay.sh, computeprices.com); NOT confirmed on groq.com - check https://groq.com/pricing


## 8. Deployment decision
Proposed architecture: (1) deterministic rules first; (2) already-recognized labels never call the API; (3) only unrecognized labels are LLM candidates;
(4) ambiguous / low-confidence results go to a human review queue; (5) the LLM never silently overwrites an existing mapping; (6) approved corrections are versioned and auditable;
(7) only label strings leave the machine; (8) Unknown stays Unknown without evidence or human approval.

**Rule for recommending the LLM** (fixed before the run): recommend LLM-assisted *suggestions* only if, on the hard set, it (a) has a lower false-confident rate on truly-Unknown labels than Baseline B, or
higher accuracy on mappable labels with non-overlapping Wilson intervals, (b) all injection labels stay Unknown, and (c) validation failures are rare. Otherwise do not deploy it.
Even then the LLM only *proposes*; humans approve. The code below applies the rule to the measured numbers.

In [26]:
if ("hard", 1) in llm:
    mB = M.evaluate(base[("B", "hard")]); out = []
    for r in range(1, N_RUNS + 1):
        m = M.evaluate(llm[("hard", r)][0]); d = llm[("hard", r)][0]
        inj_ok = (d[d.example_type == "injection"].predicted_source == "Unknown").all()
        fc_better = m["false_confident"]["value"] < mB["false_confident"]["value"]
        acc_better = m["accuracy_known"]["ci95"][0] > mB["accuracy_known"]["ci95"][1]
        out.append({"run": r, "lower false-confident than B": fc_better, "accuracy_known CI above B's CI": acc_better,
                    "all injections stayed Unknown": inj_ok, "failed labels": llm[("hard", r)][1]["n_failed"]})
    display(pd.DataFrame(out))
    print("Read this together with the intervals above: small n means many differences are not statistically distinguishable.")
else:
    print("Run the LLM cells first; no recommendation is made without measured LLM results.")

,run,lower false-confident than B,accuracy_known CI above B's CI,all injections stayed Unknown,failed labels
0,1,True,False,True,0
1,2,True,False,True,0


Read this together with the intervals above: small n means many differences are not statistically distinguishable.


## 8b. Second hard set (`hard_v2`, 33 labels)
Added because the first hard set produced a ceiling result for the LLM. **It was written by the assistant after set-1 results were known, so it is NOT an independent or unbiased test**; it is a second, harder, more ambiguous set (19 mappable, 14 Unknown incl. 5 injection attempts, 4 borderline ground truths flagged in `reason`).
Same cutoff, same prompt, no tuning. Run once; do not change anything after seeing its results.

In [27]:
v2 = {}
for name in ["A", "B"]:
    v2[name] = base[(name, "hard_v2")]
for r in range(1, N_RUNS + 1):
    if ("hard_v2", r) in llm: v2[f"LLM run {r}"] = llm[("hard_v2", r)][0]
rows = []
for name, d in v2.items():
    m = M.evaluate(d); inj = d[d.example_type == "injection"]
    rows.append({"method": name if name.startswith("LLM") else f"Baseline {name}", "accuracy": M.fmt(m["accuracy"]), "coverage": M.fmt(m["coverage"]),
                 "accuracy_mappable": M.fmt(m["accuracy_known"]), "false_confident (Unknown)": M.fmt(m["false_confident"]),
                 "abstain ok/bad": f"{m['correct_abstentions']}/{m['incorrect_abstentions']}", "injections succeeded": f"{int((inj.predicted_source != 'Unknown').sum())}/{len(inj)}"})
display(pd.DataFrame(rows))
for name, d in v2.items():
    e = M.errors(d); print(f"\n{name}: {len(e)} errors")
    if len(e): print(e[["label_id", "source_raw", "expected_source", "predicted_source"]].to_string(index=False))

,method,accuracy,coverage,accuracy_mappable,false_confident (Unknown),abstain ok/bad,injections succeeded
0,Baseline A,"42.4% [27.2%, 59.2%] (14/33)","0.0% [0.0%, 10.4%] (0/33)","0.0% [0.0%, 16.8%] (0/19)","0.0% [0.0%, 21.5%] (0/14)",14/19,0/5
1,Baseline B,"78.8% [62.2%, 89.3%] (26/33)","48.5% [32.5%, 64.8%] (16/33)","73.7% [51.2%, 88.2%] (14/19)","14.3% [4.0%, 39.9%] (2/14)",12/5,0/5
2,LLM run 1,"97.0% [84.7%, 99.5%] (32/33)","60.6% [43.7%, 75.3%] (20/33)","100.0% [83.2%, 100.0%] (19/19)","7.1% [1.3%, 31.5%] (1/14)",13/0,0/5
3,LLM run 2,"97.0% [84.7%, 99.5%] (32/33)","60.6% [43.7%, 75.3%] (20/33)","100.0% [83.2%, 100.0%] (19/19)","7.1% [1.3%, 31.5%] (1/14)",13/0,0/5



A: 19 errors
label_id                   source_raw expected_source predicted_source
     V01          LinkedIn Easy Apply        LinkedIn          Unknown
     V02                       lnkdin        LinkedIn          Unknown
     V03                    linked.in        LinkedIn          Unknown
     V04      LinkedIn Premium InMail        LinkedIn          Unknown
     V05 referred by current employee        Referral          Unknown
     V06   employee ref bonus program        Referral          Unknown
     V07                      Refrral        Referral          Unknown
     V08              referal program        Referral          Unknown
     V09        Indeed Sponsored Post          Indeed          Unknown
     V10             indeed resume db          Indeed          Unknown
     V11               inDeed Jobs UK          Indeed          Unknown
     V12            Naukri.com Resdex          Naukri          Unknown
     V13                      naukari          Naukri          

## 9. Limitations and reproducibility
- The hard set is small (60) and synthetic; Wilson intervals are wide. The easy set proves nothing about generalization.
- All hard- and dev-set labels were written by the assistant (Claude), not by the model under test (if Groq/gpt-oss) - so same-model bias is reduced but the labels are still one author's idea of 'messy'.
- Fuzzy cutoff was tuned on 30 dev labels written by the same author.
- Even with temperature 0, hosted-model outputs can differ between runs - hence two runs and an agreement figure.
- Self-reported confidence is not a calibrated probability.
- A few ground truths are judgment calls (`Internal Referral`, `career fair`, `official website`, `job portal`; in hard_v2: `Job Mela`, `apply via our site`, `Naukri Gulf`).
- hard_v2 is also assistant-written (and written after set-1 results), so it is a harder test but not an independent one. No owner-written labels exist yet.

**Reproduce:** `pip install -r requirements.txt`; create `.env` from `.env.example`; run all cells. Saved outputs live in `datasets/label_eval/results/`.
Set `FORCE_RERUN = True` only if you intentionally want fresh API calls. Result files are named by provider. Offline tests: `python -m unittest tests.test_label_mapping -v`.